# 08 Production Handoff and Release Gates (DSPy, 2026)

## What This Lesson Is
Package deployment metadata and enforce release gates before production handoff.

## Scientific Lens
- Concept: Release governance for LLM program deployment
- Measure: Gate compliance rate and handoff completeness
- Validity Limit: Checklist compliance does not guarantee runtime reliability without canary monitoring.


## How It Works
1. Create deterministic release manifest.
2. Enforce gate assertions.
3. Generate live handoff summary with LM.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Production handoff lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import json

manifest = {
    "program_version": "2026.02.19",
    "owner": "ai-platform",
    "quality_gate": {"accuracy_min": 0.80, "latency_p95_max_ms": 1200},
    "rollout": ["canary_5pct", "observe_24h", "promote_100pct"],
}

assert manifest["quality_gate"]["accuracy_min"] >= 0.8
assert manifest["quality_gate"]["latency_p95_max_ms"] <= 1500
print(json.dumps(manifest, indent=2))


In [ ]:
# Live Demo
import json
import os
from pathlib import Path

manifest = {
    "program_version": "2026.02.19",
    "owner": "ai-platform",
    "quality_gate": {"accuracy_min": 0.80, "latency_p95_max_ms": 1200},
}

try:
    import dspy
except Exception as exc:
    print(f"Skipping live handoff summary: dspy unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live handoff summary: OPENAI_API_KEY not set.")
    else:
        dspy.configure(lm=dspy.LM("openai/gpt-4.1-mini", api_key=api_key, temperature=0))
        summarize = dspy.Predict("manifest_json -> release_summary")
        out = summarize(manifest_json=json.dumps(manifest))
        manifest["release_summary"] = out.release_summary
        print(out.release_summary)

path = Path('/tmp/dspy_release_manifest.json')
path.write_text(json.dumps(manifest, indent=2))
print(path)
assert path.exists()


## Applied Labs
1. Add rollback criteria and verify they are present before release.
2. Attach owner approval metadata and fail handoff if missing.
3. Run a mock canary decision step using collected metrics.

## Validation Checklist
- Manifest includes version, owner, and explicit quality gates.
- Release gates are machine-checked, not only documented.
- Handoff artifact is persisted for downstream deployment workflows.

## Further Reading
- [DSPy Deployment Concepts](https://dspy.ai/)
- [SRE Release Engineering](https://sre.google/sre-book/release-engineering/)
- [Canary Analysis Basics](https://martinfowler.com/bliki/CanaryRelease.html)
